# Stream 2 (continued): TFT, a deep learning forecaster

**Project:** Stockout-Aware Demand Forecasting for Grocery Retail (MIS 382N)

**What this notebook adds:** the same raw vs. corrected experiment as `stream2_forecasting.ipynb`, but with **TFT (Temporal Fusion Transformer)**, a neural network built for forecasting. We train it twice, once on recorded sales (**raw**) and once on corrected demand (**corrected**), and compare it to the LightGBM results.

**TFT in plain English:** a neural network that reads the last 4 weeks of a product's history, plus things we know in advance for the days being forecast (discounts, holidays, weather, day of week), and uses *attention* to decide which past days matter most. It outputs a forecast range (10th, 50th and 90th percentile), not a single number.

**How we keep the comparison fair**
- Same 2,000 products, same date splits, same scoring (in-stock days only, against recorded sales).
- **Every forecast is made 7 days ahead.** LightGBM used history at least 7 days back, so TFT forecasts each day from an origin exactly 7 days earlier. We keep only the 7th-day-ahead output of each forecast window, so TFT sees no more recent sales than LightGBM did.
- TFT *does* also get the discount/holiday/weather values for the days in between, which LightGBM didn't. Mention that when comparing.
- TFT's main forecast is the **median** (50th percentile). LightGBM's was a mean. Medians run a bit lower than means for skewed sales, so **bias is not perfectly comparable** between the two. WAPE and MAE are.

**Before you run (important)**
1. Runtime → Change runtime type → **T4 GPU**.
2. Run the setup and install cells. If Colab asks to **restart the session** after the install, do it, then run from the imports cell down (skip the install cell).
3. Leave `QUICK_TEST = True` for the first run (about 5 minutes). It uses a few hundred products and 2 epochs, just to prove everything works. Then set `QUICK_TEST = False` for the real run.

## Step 0: Setup
Same as the first notebook: connect Drive, pull the team repo, then install the TFT library.

In [1]:
# COLAB_ONLY
from google.colab import drive, userdata
import os
drive.mount('/content/drive')

token = userdata.get('GITHUB_TOKEN')
repo = 'stockout-aware-forecasting'
if not os.path.exists(f'/content/{repo}'):
    !git clone https://{token}@github.com/bulut19/{repo}.git /content/{repo}
%cd /content/{repo}
!git pull

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/stockout-aware-forecasting
Already up to date.


In [2]:
# COLAB_ONLY
# Install the TFT library (pytorch-forecasting). This takes a minute or two.
# If Colab says "Restart session", do it, then skip this cell and rerun from the imports cell.
!pip install -q pytorch-forecasting

In [3]:
# Quick check that the GPU is on. If it says "cpu", go to Runtime -> Change runtime type -> T4 GPU
import torch
print("Using:", "GPU (" + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "cpu  <-- turn on the GPU!")

Using: GPU (Tesla T4)


In [4]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
warnings.filterwarnings('ignore')

# Shared team code, then point PROC_DIR at the folder where the data really lives in YOUR Drive
from src.load_data import DRIVE_DIR, PROC_DIR
from pathlib import Path
PROC_DIR = Path('/content/drive/MyDrive/Final Project/Stockout Project Data/processed')

# ---------------- Settings (all in one place) ----------------
QUICK_TEST   = False     # True = tiny practice run to check it works. Set to False for the real run.
N_PRODUCTS   = 300      # products used when QUICK_TEST is True
MAX_EPOCHS   = 2 if QUICK_TEST else 15
ENCODER_LEN  = 28       # how many past days TFT reads (4 weeks)
HORIZON      = 7        # forecast 7 days ahead
HIDDEN       = 16       # size of the network (small on purpose: our data is small)
BATCH        = 256
SEED         = 42
NUM_WORKERS  = 2
CKPT_DIR     = PROC_DIR.parent / 'tft_checkpoints'   # best models are saved here, so a disconnect doesn't lose them

pl.seed_everything(SEED)
torch.set_float32_matmul_precision('medium')
print("QUICK_TEST =", QUICK_TEST)

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


QUICK_TEST = False


## Step 1: Load the data
The same two files as before: corrected demand (`daily_demand_v1`) and the context columns (discount, weather, holidays) from `daily_model`.

In [5]:
sample_ids = pd.read_csv(PROC_DIR / 'sample_ids.csv').series_id.tolist()
if QUICK_TEST:
    # Practice run: just the first few hundred products so it's fast
    sample_ids = sample_ids[:N_PRODUCTS]

dd = pd.read_parquet(PROC_DIR / 'daily_demand_v1.parquet')
dd['dt'] = pd.to_datetime(dd['dt'])
dd = dd[dd.series_id.isin(sample_ids)]

context_cols = ['series_id', 'dt', 'day_of_week', 'discount', 'holiday_flag', 'activity_flag',
                'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level', 'first_category_id']
ctx = pd.read_parquet(PROC_DIR / 'daily_model.parquet', columns=context_cols)
ctx['dt'] = pd.to_datetime(ctx['dt'])
ctx = ctx[ctx.series_id.isin(sample_ids)]

base = dd.merge(ctx, on=['series_id', 'dt'], how='left').sort_values(['series_id', 'dt']).reset_index(drop=True)

# Day number for each date: 0 for the first day, 96 for the last. TFT works in day numbers.
base['time_idx'] = (base.dt - base.dt.min()).dt.days

# Where each split starts and ends, as day numbers (train 0-75, valid 76-89, test 90-96)
TRAIN_END = int(base[base.set == 'train'].time_idx.max())
VALID_END = int(base[base.set == 'valid'].time_idx.max())
TEST_END  = int(base[base.set == 'test'].time_idx.max())
print(f"{base.series_id.nunique():,} products | train days 0-{TRAIN_END}, valid {TRAIN_END+1}-{VALID_END}, test {VALID_END+1}-{TEST_END}")

# Every product must have every day (TFT needs unbroken time series)
assert base.groupby('series_id').size().nunique() == 1, "Some products are missing days!"

2,000 products | train days 0-75, valid 76-89, test 90-96


## Step 2: Prepare the inputs for TFT
- TFT wants category-like columns as text, so day of week and product category become text.
- Missing discounts (marked as missing in the data) are filled with 1.0, meaning "no discount."
- We build **two versions** of the table. They are identical except for the target (and its history): `sales` for **raw**, `demand` for **corrected**.

In [6]:
KNOWN_REALS = ['discount', 'holiday_flag', 'activity_flag', 'precpt',
               'avg_temperature', 'avg_humidity', 'avg_wind_level']

base['discount'] = base['discount'].fillna(1.0)
for c in KNOWN_REALS:
    base[c] = base[c].astype('float32').fillna(base[c].mean())

base['day_of_week_cat'] = base['day_of_week'].astype(int).astype(str)      # TFT wants text categories
base['category_cat']    = base['first_category_id'].astype(int).astype(str)

def make_version(df, value_col):
    d = df.copy()
    d['target'] = d[value_col].astype('float32')    # what TFT learns to predict (and reads as history)
    return d

data = {'raw': make_version(base, 'sales'), 'corrected': make_version(base, 'demand')}

# Stockout group for each product (low / medium / high), from TRAINING days only, same as before
tr = base[base.set == 'train']
rate = (tr.oos_hours > 0).groupby(tr.series_id).mean()
band_map = pd.qcut(rate.rank(method='first'), 3, labels=['low', 'medium', 'high']).astype(str).to_dict()
base['stockout_band'] = base.series_id.map(band_map)
print("Ready. Rows per version:", len(data['raw']))

Ready. Rows per version: 194000


## Step 3: Build the training windows
TFT learns from many short "windows": show it the last up to 28 days of a product's history, then ask it to forecast the next 7 days. We slide that window across the training period.

Three sets of windows:
- **Training windows:** the forecast period is inside the training days only.
- **Validation windows:** the forecast period is inside the validation days. Used only to decide when to stop training.
- **Scoring windows:** used at the end to get a forecast for every validation and test day. For each day, the window started 7 days earlier, so the forecast really is 7 days ahead.

In [7]:
def make_datasets(df):
    # --- Training windows: forecast days are all within the training period
    training = TimeSeriesDataSet(
        df[df.time_idx <= TRAIN_END],
        time_idx='time_idx', target='target', group_ids=['series_id'],
        min_encoder_length=14, max_encoder_length=ENCODER_LEN,        # reads 2-4 weeks of history
        min_prediction_length=HORIZON, max_prediction_length=HORIZON, # forecasts 7 days
        static_categoricals=['category_cat'],                         # facts that never change
        time_varying_known_categoricals=['day_of_week_cat'],          # known in advance
        time_varying_known_reals=KNOWN_REALS,                         # known in advance
        time_varying_unknown_reals=['target'],                        # the sales history itself
        target_normalizer=GroupNormalizer(groups=['series_id'], transformation='softplus'),  # puts each product on its own scale
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
        allow_missing_timesteps=False)

    # --- Validation windows: forecast days fall in the validation period
    validation = TimeSeriesDataSet.from_dataset(
        training, df[df.time_idx <= VALID_END], predict=False, stop_randomization=True,
        min_prediction_idx=TRAIN_END + 1, min_encoder_length=ENCODER_LEN)   # always a full 4 weeks of history

    # --- Scoring windows: every window whose 7th forecast day lands on a valid or test day
    scoring = TimeSeriesDataSet.from_dataset(
        training, df, predict=False, stop_randomization=True,
        min_prediction_idx=TRAIN_END + 1 - (HORIZON - 1), min_encoder_length=ENCODER_LEN)   # exactly one window per product per day
    return training, validation, scoring

datasets = {v: make_datasets(df) for v, df in data.items()}
for v, (tr_ds, va_ds, sc_ds) in datasets.items():
    print(f"{v:>9}: {len(tr_ds):,} training windows | {len(va_ds):,} validation | {len(sc_ds):,} scoring")

      raw: 140,000 training windows | 16,000 validation | 42,000 scoring
corrected: 140,000 training windows | 16,000 validation | 42,000 scoring


## Step 4: Train TFT (raw, then corrected)
Both versions use **exactly the same settings**. Only the data differs.

- **Loss:** quantile loss, so TFT learns the 10th, 50th and 90th percentile forecasts directly (the ranges stream 5 needs).
- **Early stopping:** training stops when the validation loss stops improving for 3 epochs, so it doesn't memorize noise.
- **Checkpoints:** the best version of the model is saved to Drive after each improvement. If Colab disconnects, you can reload it instead of retraining.
- **Time:** a few minutes per epoch for the full run on a T4 (my estimate). The practice run is much faster.

In [8]:
def train_tft(version):
    training, validation, _ = datasets[version]
    train_dl = training.to_dataloader(train=True,  batch_size=BATCH, num_workers=NUM_WORKERS)
    val_dl   = validation.to_dataloader(train=False, batch_size=BATCH * 2, num_workers=NUM_WORKERS)

    model = TemporalFusionTransformer.from_dataset(
        training, learning_rate=0.03, hidden_size=HIDDEN, attention_head_size=2, dropout=0.1,
        hidden_continuous_size=8, loss=QuantileLoss(quantiles=[0.1, 0.5, 0.9]),
        optimizer='adam', reduce_on_plateau_patience=2, log_interval=-1)
    print(f"[{version}] model has {sum(p.numel() for p in model.parameters()):,} learnable numbers")

    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    ckpt = ModelCheckpoint(dirpath=str(CKPT_DIR / version), filename='best', monitor='val_loss', mode='min', save_top_k=1)
    stop = EarlyStopping(monitor='val_loss', patience=3, mode='min')

    trainer = pl.Trainer(max_epochs=MAX_EPOCHS, accelerator='auto', gradient_clip_val=0.1,
                         callbacks=[stop, ckpt], logger=False, enable_model_summary=False)
    trainer.fit(model, train_dl, val_dl)

    best = TemporalFusionTransformer.load_from_checkpoint(ckpt.best_model_path)   # reload the best version
    print(f"[{version}] best validation loss: {float(ckpt.best_model_score):.4f}")
    return best

tft_models = {}
tft_models['raw'] = train_tft('raw')

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[raw] model has 27,435 learnable numbers


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

[raw] best validation loss: 0.2595


In [9]:
tft_models['corrected'] = train_tft('corrected')

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

[corrected] model has 27,435 learnable numbers


[corrected] best validation loss: 0.2484


## Step 5: Make the forecasts
For every product and every validation/test day we get the 10th, 50th and 90th percentile. We keep only the **7th-day-ahead** output of each window, so every forecast is made exactly 7 days ahead, matching LightGBM's setup. The median (50th) is the main forecast.

In [ ]:
start_date = base.dt.min()
lookup = base[['series_id', 'dt', 'set', 'sales', 'demand', 'oos_hours', 'stockout_band', 'first_category_id']]

def forecast_table(version):
    _, _, scoring = datasets[version]
    dl = scoring.to_dataloader(train=False, batch_size=BATCH * 4, num_workers=NUM_WORKERS)
    pred = tft_models[version].predict(dl, mode='quantiles', return_index=True,
                                       trainer_kwargs=dict(accelerator='auto', logger=False))
    q = pred.output[:, -1, :].cpu().numpy()          # last step = the 7th day ahead; columns = q10, q50, q90
    idx = pred.index.copy()                           # which product / start day each window belongs to
    out = pd.DataFrame({'series_id': idx.series_id.values,
                        'dt': start_date + pd.to_timedelta(idx.time_idx.values + HORIZON - 1, unit='D'),
                        'q10': q[:, 0], 'q50': q[:, 1], 'q90': q[:, 2]})
    out[['q10', 'q50', 'q90']] = np.sort(out[['q10', 'q50', 'q90']].to_numpy(), axis=1).clip(min=0)  # keep low <= mid <= high, never negative
    out['forecast'] = out['q50']                      # main forecast = the median
    out = out.merge(lookup, on=['series_id', 'dt'], how='left')
    out = out[out.set.isin(['valid', 'test'])]
    out['version'] = version
    out['model'] = 'tft'
    return out

tft_forecasts = pd.concat([forecast_table(v) for v in ['raw', 'corrected']], ignore_index=True)
tft_forecasts = tft_forecasts[['series_id', 'dt', 'set', 'sales', 'demand', 'oos_hours', 'stockout_band',
                               'first_category_id', 'version', 'model', 'forecast', 'q10', 'q50', 'q90']]

# Safety checks: one forecast per product per day, nothing missing
expected = base.series_id.nunique() * (VALID_END - TRAIN_END + TEST_END - VALID_END)
for v, g in tft_forecasts.groupby('version'):
    assert len(g) == expected, f"{v}: {len(g)} rows, expected {expected}"
    assert not g.duplicated(['series_id', 'dt']).any()
    assert g.forecast.notna().all()
print("Every product has a TFT forecast for every valid and test day.")
tft_forecasts.groupby('version')[['forecast', 'q10', 'q50', 'q90']].mean().round(3)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

## Step 6: Score TFT and compare with the other models
Same rules as before: **in-stock days only**, against recorded sales. Lower WAPE and MAE are better, and bias closer to 0 is better (negative = under-forecasting).

We load the earlier seasonal naive and LightGBM forecasts and put TFT next to them. *Remember: TFT's forecast is a median and LightGBM's is a mean, so compare bias with care.*

In [ ]:
def wape(actual, pred): return float(np.abs(actual - pred).sum() / np.abs(actual).sum())
def mae(actual, pred):  return float(np.abs(actual - pred).mean())
def bias(actual, pred): return float((pred - actual).sum() / actual.sum())   # negative = forecasts too low
def pinball(actual, pred_q, q):
    diff = actual - pred_q
    return float(np.mean(np.maximum(q * diff, (q - 1) * diff)))

def score_table(fc, which_set, group_cols=('model', 'version')):
    s = fc[(fc.set == which_set) & (fc.oos_hours == 0)]      # in-stock days only
    rows = []
    for key, g in s.groupby(list(group_cols)):
        key = key if isinstance(key, tuple) else (key,)
        rows.append({**dict(zip(group_cols, key)), 'wape': wape(g.sales, g.forecast),
                     'mae': mae(g.sales, g.forecast), 'bias': bias(g.sales, g.forecast), 'n_days': len(g)})
    return pd.DataFrame(rows)

# Earlier models' forecasts (limited to the same products, in case this is the practice run)
earlier = pd.read_parquet(PROC_DIR / 'forecasts_stream2_v1.parquet')
earlier = earlier[earlier.series_id.isin(tft_forecasts.series_id.unique())]
all_fc = pd.concat([earlier, tft_forecasts], ignore_index=True)

valid_scores = score_table(all_fc, 'valid')
test_scores  = score_table(all_fc, 'test')
print("VALIDATION (in-stock days)"); display(valid_scores.round(3))
print("TEST (in-stock days)");       display(test_scores.round(3))

In [ ]:
# Side by side: how much did correcting change each model?
test_scores.pivot(index='model', columns='version', values=['wape', 'mae', 'bias']).round(3)

In [ ]:
# Forecast ranges for TFT: pinball loss (lower is better) and coverage (ideal: about 80%)
t = tft_forecasts[(tft_forecasts.set == 'test') & (tft_forecasts.oos_hours == 0)]
rows = []
for version, g in t.groupby('version'):
    rows.append({'version': version, 'pinball_q10': pinball(g.sales, g.q10, 0.1),
                 'pinball_q50': pinball(g.sales, g.q50, 0.5), 'pinball_q90': pinball(g.sales, g.q90, 0.9),
                 'coverage_10_to_90': float(((g.sales >= g.q10) & (g.sales <= g.q90)).mean())})
pd.DataFrame(rows).round(3)

In [ ]:
# Does TFT help more for products that stock out a lot? (TFT only, test week)
by_band = score_table(tft_forecasts, 'test', group_cols=('stockout_band', 'version'))
by_band['stockout_band'] = pd.Categorical(by_band.stockout_band, ['low', 'medium', 'high'])
by_band.sort_values(['stockout_band', 'version']).round(3)

## Step 7: How stable is it? (confidence check)
Same idea as in the first notebook: re-grade the test week 2,000 times, each time with a random draw of products (with repeats). If a model wins in nearly every draw, the result is stable across products. It does **not** cover other weeks.

In [ ]:
N_DRAWS = 2000
rng = np.random.default_rng(SEED)

t = all_fc[(all_fc.set == 'test') & (all_fc.oos_hours == 0)].copy()
t['abs_err']    = (t.sales - t.forecast).abs()
t['signed_err'] = t.forecast - t.sales
per_product = t.groupby(['model', 'version', 'series_id'])[['abs_err', 'signed_err', 'sales']].sum()

all_products = np.sort(t.series_id.unique())
draws = rng.integers(0, len(all_products), size=(N_DRAWS, len(all_products)))   # same draws for every model

def draw_scores(model, version):
    p = per_product.loc[(model, version)].reindex(all_products, fill_value=0.0)
    a, s, y = p.abs_err.to_numpy(), p.signed_err.to_numpy(), p.sales.to_numpy()
    return a[draws].sum(axis=1) / y[draws].sum(axis=1), s[draws].sum(axis=1) / y[draws].sum(axis=1)

models_list = ['seasonal_naive', 'lightgbm', 'tft']
scores = {(m, v): draw_scores(m, v) for m in models_list for v in ['raw', 'corrected']}

rows = [{'model': m, 'version': v, 'wape': w.mean(), 'wape_low': np.percentile(w, 2.5), 'wape_high': np.percentile(w, 97.5)}
        for (m, v), (w, b) in scores.items()]
print("Each model's test-week error, with a 95% range across product draws")
display(pd.DataFrame(rows).round(3))

def compare(label, first, second):
    w1, b1 = scores[first]; w2, b2 = scores[second]
    diff = w1 - w2                                  # negative = the first model has lower error
    lo, hi = np.percentile(diff, [2.5, 97.5])
    return {'comparison': label, 'wape_gap': diff.mean(), 'ci_low': lo, 'ci_high': hi,
            'share_first_better': (diff < 0).mean(), 'bias_closer_to_0': (np.abs(b1) < np.abs(b2)).mean()}

stability = pd.DataFrame([
    compare('TFT: corrected vs raw',                 ('tft', 'corrected'), ('tft', 'raw')),
    compare('Raw: TFT vs LightGBM',                  ('tft', 'raw'),       ('lightgbm', 'raw')),
    compare('Corrected: TFT vs LightGBM',            ('tft', 'corrected'), ('lightgbm', 'corrected')),
    compare('Corrected: TFT vs seasonal naive',      ('tft', 'corrected'), ('seasonal_naive', 'corrected')),
    compare('Corrected TFT vs raw LightGBM',         ('tft', 'corrected'), ('lightgbm', 'raw')),
])
print("\nHead-to-head (negative gap = the first model is better)")
display(stability.round(3))

## Step 8: Chart
Test-week forecasts for one example product: the TFT forecast with its 10th to 90th range, trained on raw sales vs. corrected demand. Red X marks a day with an empty shelf.

In [ ]:
os.makedirs('figures', exist_ok=True)
tf = tft_forecasts[tft_forecasts.set.isin(['valid', 'test'])]
# Pick a product with a decent number of stockout days but not the most extreme one
per = tf[tf.version == 'raw'].groupby('series_id').agg(so=('oos_hours', lambda s: (s > 0).sum()), avg=('sales', 'mean'))
cand = per[(per.avg >= per.avg.median()) & (per.so >= per.so.quantile(0.5))]
example = (cand.so - cand.so.median()).abs().idxmin() if len(cand) else per.so.idxmax()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, version in zip(axes, ['raw', 'corrected']):
    e = tf[(tf.series_id == example) & (tf.version == version)].sort_values('dt')
    ax.fill_between(e.dt, e.q10, e.q90, alpha=0.25, label='TFT 10th to 90th range')
    ax.plot(e.dt, e.q50, lw=2, label='TFT forecast (median)')
    ax.plot(e.dt, e.sales, color='black', lw=1.3, label='Recorded sales')
    so = e[e.oos_hours > 0]
    ax.scatter(so.dt, so.sales, color='red', marker='x', s=50, zorder=5, label='Day with empty shelf')
    ax.set_title(f'Trained on {version}'); ax.tick_params(axis='x', rotation=30)
axes[0].set_ylabel('Daily sales (scaled units)'); axes[0].legend(fontsize=8, loc='upper left')
fig.suptitle(f'TFT forecasts, example product {example}'); plt.tight_layout()
plt.savefig('figures/stream2_tft_example.png', dpi=150); plt.show()

### How to read the results
- **TFT trained on raw vs. corrected** is the main comparison here, the same question as for LightGBM.
- **TFT vs. LightGBM** shows whether the deep learning model earns its extra complexity on this small dataset. It's fine if it doesn't: with 76 training days per product, simpler models often win.
- TFT forecasts a **median**, so it may show more negative bias than LightGBM even if it is just as good. Judge TFT mostly by WAPE and MAE.
- Corrected demand is an **estimate**, not the truth. Report cases where correcting does not help.

## Step 9: Save for the team
New files, so nothing from the first notebook is overwritten. Only run this with `QUICK_TEST = False`, otherwise the files would contain only the practice products.
- **`forecasts_stream2_tft_v1.parquet`**: same columns as `forecasts_stream2_v1.parquet`, with `model = 'tft'`. (`q50` equals `forecast`.)
- **`results_stream2_tft_v1.csv`**: the score tables.
- **`stability_stream2_tft_v1.csv`**: the confidence check.

In [ ]:
if QUICK_TEST:
    print("QUICK_TEST is on, so nothing was saved. Set QUICK_TEST = False and rerun the notebook for the real run.")
else:
    tft_forecasts.to_parquet(PROC_DIR / 'forecasts_stream2_tft_v1.parquet', index=False)
    pd.concat([valid_scores.assign(set='valid'), test_scores.assign(set='test')]).to_csv(PROC_DIR / 'results_stream2_tft_v1.csv', index=False)
    stability.to_csv(PROC_DIR / 'stability_stream2_tft_v1.csv', index=False)
    print("Saved forecasts_stream2_tft_v1.parquet, results_stream2_tft_v1.csv, stability_stream2_tft_v1.csv")

## Reuse and disclosure notes (for the final report)
- **Library used (not a pretrained model):** `pytorch-forecasting` (provides the TFT architecture), PyTorch and PyTorch Lightning. TFT is from Lim et al. (2021), *Temporal Fusion Transformers for Interpretable Multi-horizon Time Series Forecasting*. We trained it from scratch on our data.
- **Pretrained models used:** none.
- **LLM use:** this notebook was written with AI assistance.

In [ ]:
# COLAB_ONLY
# Save this notebook to the team's GitHub repo on its own branch. Fill in your details first,
# and make sure the notebook is saved in Drive (File -> Save).
%cd /content/stockout-aware-forecasting
NAME  = 'your-github-username'
EMAIL = 'the-email-on-your-github-account'
NOTEBOOK_PATH = '/content/drive/MyDrive/Final Project/stream2_tft.ipynb'   # change if you saved it elsewhere

!git config user.name "{NAME}"
!git config user.email "{EMAIL}"
!git checkout -B stream2-tft
!cp "{NOTEBOOK_PATH}" notebooks/stream2_tft.ipynb
!mkdir -p figures && git add notebooks/stream2_tft.ipynb figures/stream2_tft_*.png
!git commit -m "Stream 2: TFT, raw vs corrected demand"
!git push -u origin stream2-tft